# **Khởi tạo**

In [1]:
# Import
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Xác định thư mục gốc project
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Tạo thư mục nếu chưa có
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

BASE_DIR: d:\Khai phá dữ liệu\DoAn\traffic-accidents-mining
RAW_DIR: d:\Khai phá dữ liệu\DoAn\traffic-accidents-mining\data\raw
PROCESSED_DIR: d:\Khai phá dữ liệu\DoAn\traffic-accidents-mining\data\processed


In [3]:
# Load dataset
file_path = RAW_DIR / "traffic_accidents.csv"

df = pd.read_csv(file_path)

print("Loaded file:", file_path)
df.head()

Loaded file: d:\Khai phá dữ liệu\DoAn\traffic-accidents-mining\data\raw\traffic_accidents.csv


,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
0,07/29/2023 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,13,7,7
1,08/13/2023 12:11:00 AM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,0,1,8
2,12/09/2021 10:30:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,10,5,12
3,08/09/2023 07:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,5.0,0.0,0.0,5.0,0.0,0.0,19,4,8
4,08/19/2023 02:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,14,7,8


In [4]:
# Xem dataset
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())

Shape: (209306, 24)

Columns:
 ['crash_date', 'traffic_control_device', 'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'alignment', 'roadway_surface_cond', 'road_defect', 'crash_type', 'intersection_related_i', 'damage', 'prim_contributory_cause', 'num_units', 'most_severe_injury', 'injuries_total', 'injuries_fatal', 'injuries_incapacitating', 'injuries_non_incapacitating', 'injuries_reported_not_evident', 'injuries_no_indication', 'crash_hour', 'crash_day_of_week', 'crash_month']


In [5]:
# Info + missing
df.info()
df.isnull().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209306 entries, 0 to 209305
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   crash_date                     209306 non-null  object 
 1   traffic_control_device         209306 non-null  object 
 2   weather_condition              209306 non-null  object 
 3   lighting_condition             209306 non-null  object 
 4   first_crash_type               209306 non-null  object 
 5   trafficway_type                209306 non-null  object 
 6   alignment                      209306 non-null  object 
 7   roadway_surface_cond           209306 non-null  object 
 8   road_defect                    209306 non-null  object 
 9   crash_type                     209306 non-null  object 
 10  intersection_related_i         209306 non-null  object 
 11  damage                         209306 non-null  object 
 12  prim_contributory_cause       

crash_date                       0
traffic_control_device           0
weather_condition                0
lighting_condition               0
first_crash_type                 0
trafficway_type                  0
alignment                        0
roadway_surface_cond             0
road_defect                      0
crash_type                       0
intersection_related_i           0
damage                           0
prim_contributory_cause          0
num_units                        0
most_severe_injury               0
injuries_total                   0
injuries_fatal                   0
injuries_incapacitating          0
injuries_non_incapacitating      0
injuries_reported_not_evident    0
injuries_no_indication           0
crash_hour                       0
crash_day_of_week                0
crash_month                      0
dtype: int64

In [6]:
# Check giá trị UNIQUE
unique_df = pd.DataFrame({
    "column": df.columns,
    "n_unique": df.nunique().values
}).sort_values(by="n_unique", ascending=False).reset_index(drop=True)

display(unique_df)

,column,n_unique
0,crash_date,189087
1,prim_contributory_cause,40
2,injuries_no_indication,39
3,crash_hour,24
4,trafficway_type,20
5,injuries_total,19
6,traffic_control_device,19
7,injuries_non_incapacitating,19
8,first_crash_type,18
9,injuries_reported_not_evident,13


# **Tiền xử lý dữ liệu**

## 1. Chuẩn hóa dữ liệu nền

Mục tiêu của bước này:
- Tạo một bản dữ liệu làm việc riêng để không làm thay đổi dữ liệu gốc
- Chuẩn hóa các cột dạng text để tránh sai khác do khoảng trắng, chữ hoa/thường
- Tạo nền dữ liệu thống nhất cho các phần: EDA, gom cụm, phân loại và luật kết hợp

In [7]:
# 1. Chuẩn hóa dữ liệu nền

# Tạo bản sao để xử lý, giữ nguyên df gốc
df_clean = df.copy()

# Xác định các cột dạng text
object_cols = df_clean.select_dtypes(include="object").columns.tolist()
object_cols.remove("crash_date")

# Chuẩn hóa text:
# - chuyển sang string để xử lý đồng nhất
# - bỏ khoảng trắng thừa ở đầu/cuối
# - đưa về chữ in hoa để tránh cùng nghĩa nhưng khác cách viết
for col in object_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()

# Kiểm tra nhanh
print("Số cột text:", len(object_cols))
print("Các cột text:")
print(object_cols)

Số cột text: 13
Các cột text:
['traffic_control_device', 'weather_condition', 'lighting_condition', 'first_crash_type', 'trafficway_type', 'alignment', 'roadway_surface_cond', 'road_defect', 'crash_type', 'intersection_related_i', 'damage', 'prim_contributory_cause', 'most_severe_injury']


## 2. Chuẩn hóa missing ngầm

Mục tiêu:
- Chuẩn hóa các giá trị biểu thị thiếu thông tin về một dạng thống nhất ("UNKNOWN")
- Giữ tính nhất quán cho toàn bộ pipeline
- Hạn chế phân mảnh category trước các bước phân tích

In [9]:
# 2. Chuẩn hóa missing ngầm

# Các giá trị được xem là thiếu thông tin
hidden_missing_tokens = [
    "UNKNOWN",
    "UNABLE TO DETERMINE",
    "NOT REPORTED",
    "UNKNOWN INTERSECTION TYPE"
]

# Đếm số lượng trước khi chuẩn hóa
before_counts = {
    token: (df_clean == token).sum().sum()
    for token in hidden_missing_tokens
}

# Chuẩn hóa về "UNKNOWN"
df_clean.replace(hidden_missing_tokens, "UNKNOWN", inplace=True)

# Đếm lại sau khi chuẩn hóa
after_unknown_count = (df_clean == "UNKNOWN").sum().sum()

# In kết quả kiểm tra
print("=== Trước khi chuẩn hóa ===")
for k, v in before_counts.items():
    print(f"{k}: {v}")

print("\n=== Sau khi chuẩn hóa ===")
print(f"Tổng số 'UNKNOWN': {after_unknown_count}")

=== Trước khi chuẩn hóa ===
UNKNOWN: 63320
UNABLE TO DETERMINE: 58316
NOT REPORTED: 581
UNKNOWN INTERSECTION TYPE: 1885

=== Sau khi chuẩn hóa ===
Tổng số 'UNKNOWN': 124102


## 3. Chuẩn hóa kiểu dữ liệu

Mục tiêu:
- Đưa dữ liệu về định dạng phù hợp cho phân tích
- Chuẩn hóa các cột dạng nhị phân
- Chuẩn bị dữ liệu cho các bước xử lý tiếp theo

In [10]:
# 3. Chuẩn hóa kiểu dữ liệu

# 3.1. Chuẩn hóa biến nhị phân: Y/N -> 1/0
df_clean["intersection_related_i"] = df_clean["intersection_related_i"].map({
    "Y": 1,
    "N": 0
})

# 3.2. Các cột cần đảm bảo ở dạng số
numeric_cols = [
    "crash_hour",
    "crash_day_of_week",
    "crash_month",
    "num_units",
    "injuries_total",
    "injuries_fatal",
    "injuries_incapacitating",
    "injuries_non_incapacitating",
    "injuries_reported_not_evident",
    "injuries_no_indication"
]

# Ép kiểu numeric cho các cột số
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 3.3. Tạo bảng kiểm tra kết quả sau chuẩn hóa
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

info_df = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.astype(str).values,
    "non_null": df_clean.notna().sum().values,
    "missing": df_clean.isna().sum().values,
    "n_unique": df_clean.nunique().values
})

display(info_df)

,column,dtype,non_null,missing,n_unique
0,crash_date,object,209306,0,189087
1,traffic_control_device,object,209306,0,19
2,weather_condition,object,209306,0,12
3,lighting_condition,object,209306,0,6
4,first_crash_type,object,209306,0,18
5,trafficway_type,object,209306,0,18
6,alignment,object,209306,0,6
7,roadway_surface_cond,object,209306,0,7
8,road_defect,object,209306,0,7
9,crash_type,object,209306,0,2


## 4. Chuẩn hóa cột thời gian

Mục tiêu:
- Chuyển cột thời gian về đúng kiểu dữ liệu datetime
- Đảm bảo dữ liệu thời gian nhất quán và dễ sử dụng cho các bước phân tích tiếp theo
- Giữ nguyên các biến thời gian đã được tách sẵn trong dataset

In [11]:
# 4. Chuẩn hóa cột thời gian

df_clean["crash_date"] = pd.to_datetime(
    df_clean["crash_date"],
    errors="coerce"
)

# Kiểm tra
print("Dtype:", df_clean["crash_date"].dtype)
print("Missing sau parse:", df_clean["crash_date"].isna().sum())

display(df_clean["crash_date"].head())

C:\Users\ACER\AppData\Local\Temp\ipykernel_82460\241946047.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["crash_date"] = pd.to_datetime(


Dtype: datetime64[ns]
Missing sau parse: 0


0   2023-07-29 13:00:00
1   2023-08-13 00:11:00
2   2021-12-09 10:30:00
3   2023-08-09 19:55:00
4   2023-08-19 14:55:00
Name: crash_date, dtype: datetime64[ns]

## 5. Lưu dataset sau tiền xử lý

Mục tiêu:
- Lưu dataset đã được làm sạch và chuẩn hóa
- Đảm bảo dữ liệu có thể tái sử dụng cho các bước tiếp theo như clustering, classification và association

In [15]:
# 5. Lưu dataset sau tiền xử lý

# Dùng đúng đường dẫn gốc của project
output_path = PROCESSED_DIR / "clean_dataset.csv"

# Lưu file
df_clean.to_csv(output_path, index=False)

print("Đã lưu dataset tại:", output_path)
print("Shape:", df_clean.shape)

Đã lưu dataset tại: d:\Khai phá dữ liệu\DoAn\traffic-accidents-mining\data\processed\clean_dataset.csv
Shape: (209306, 24)
